# 01 — Build AnnData from Kinker et al. 2020 raw files

Parses `data/Metadata.txt` and `data/UMIcount_data.txt` (from Broad Single Cell Portal SCP542) into a single AnnData object.

`Metadata.txt` has one descriptor row (data type per column) before the real rows start, which we drop. `UMIcount_data.txt` is genes x cells with 3 header rows (`CellID`, `CellLine`, `Pool`); we only need `CellID` to align cells with `Metadata.txt`, since cell line / pool / cancer type are already carried per-cell in the metadata table.

In [1]:
import os
import numpy as np
import pandas as pd
import scipy.sparse
import anndata
import scanpy as sc

DATA_DIR = os.path.join(os.getcwd(), 'data')
OUTS_DIR = os.path.join(os.getcwd(), 'outs')
os.makedirs(OUTS_DIR, exist_ok=True)

In [2]:
meta = pd.read_csv(os.path.join(DATA_DIR, 'Metadata.txt'), sep='\t')
meta = meta.drop(index=0)  # first row is a column-type descriptor, not a cell
meta = meta.rename(columns={
    'NAME': 'CellID', 'Cell_line': 'CellLine', 'Pool_ID': 'Pool', 'Cancer_type': 'Indication',
    'G1/S_score': 'G1_S_score', 'G2/M_score': 'G2_M_score',  # '/' breaks h5ad (HDF5 path separator)
})
meta.head()

/var/folders/hn/zw0sbvr9603b7m_bzxhsm9080000gn/T/ipykernel_52631/1740344039.py:1: DtypeWarning: Columns (4,9,10,11,12,13,14,15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(DATA_DIR, 'Metadata.txt'), sep='\t')


,CellID,CellLine,Pool,Indication,Genes_expressed,Discrete_cluster_minpts5_eps1.8,Discrete_cluster_minpts5_eps1.5,Discrete_cluster_minpts5_eps1.2,CNA_subclone,SkinPig_score,...,EMTII_score,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1_S_score,G2_M_score
1,AAACCTGAGACATAAC-1-18,NCIH2126_LUNG,18,Lung Cancer,4318,NaN,NaN,NaN,NaN,0.166,...,-0.935,-0.935,0.13,0.619,1.869,-0.004,0.805,0.896,0.424,-1.125
2,AACGTTGTCACCCGAG-1-18,NCIH2126_LUNG,18,Lung Cancer,5200,NaN,NaN,NaN,NaN,-0.213,...,-1.027,-1.027,0.066,1.049,1.267,0.252,1.299,1.61,0.624,-0.048
3,AACTGGTAGACACGAC-1-18,NCIH2126_LUNG,18,Lung Cancer,4004,NaN,NaN,NaN,NaN,-0.101,...,-0.677,-0.677,0.304,0.822,2.401,0.141,0.451,1.225,-0.795,0.064
4,AACTGGTAGGGCTTGA-1-18,NCIH2126_LUNG,18,Lung Cancer,4295,NaN,NaN,NaN,NaN,-0.014,...,-0.735,-0.735,0.094,0.834,2.282,0.15,0.267,0.892,-0.238,1.118
5,AACTGGTAGTACTTGC-1-18,NCIH2126_LUNG,18,Lung Cancer,4842,NaN,NaN,NaN,NaN,0.006,...,-0.821,-0.821,0.034,0.96,1.4,-0.012,-0.276,-0.428,0.267,0.791


In [3]:
# UMIcount_data.txt: rows = genes, columns = cells. Row 0 = CellID header we need;
# rows 1-2 (CellLine, Pool) are redundant with Metadata.txt, so we skip them on the real read.
cell_ids = pd.read_csv(os.path.join(DATA_DIR, 'UMIcount_data.txt'), sep='\t', header=None, nrows=1).T
cell_ids = cell_ids.drop(index=0)[0]
cell_ids

1        AAACCTGAGACATAAC-1-18
2        AAACCTGCACAACGCC-1-18
3        AAACCTGCAGACAAGC-1-18
4        AAACCTGCAGCTCGAC-1-18
5        AAACCTGCATGGATGG-1-18
                 ...          
56978                    c4788
56979                    c4789
56980                    c4793
56981                    c4800
56982                    c4812
Name: 0, Length: 56982, dtype: object

In [4]:
%%time
# 30,314 genes x 53,513 cells as dense float64 would be ~13GB+ during parsing/transpose -
# too much for a 16GB laptop. Read in gene-row chunks, sparsify each chunk immediately,
# and stack - final sparse matrix is far smaller since UMI count data is mostly zeros.
# (dtype= can't be passed to read_csv here - pandas' C parser applies a scalar dtype to
# the index column too, before index_col is split off, which breaks on gene-name strings.)
chunks = []
gene_names = []
chunk_iter = pd.read_csv(
    os.path.join(DATA_DIR, 'UMIcount_data.txt'),
    sep='\t', header=None, skiprows=3, index_col=0, chunksize=2000,
)
for chunk in chunk_iter:
    gene_names.extend(chunk.index.tolist())
    chunks.append(scipy.sparse.csr_matrix(chunk.values.astype(np.int32)))

counts_sparse = scipy.sparse.vstack(chunks, format='csr')  # genes x cells
counts_sparse = counts_sparse.T.tocsr()  # cells x genes, cheap sparse transpose
del chunks
counts_sparse.shape

CPU times: user 5min 44s, sys: 17.6 s, total: 6min 2s
Wall time: 6min 6s


(56982, 30314)

In [5]:
cell_ids_arr = cell_ids.values
gene_names_arr = np.array(gene_names)

# Keep only cells present in both files, and align metadata to the count matrix's cell order
mask = pd.Series(cell_ids_arr).isin(meta['CellID']).values
counts_sparse = counts_sparse[mask, :]
cell_ids_arr = cell_ids_arr[mask]

meta = meta.set_index('CellID').reindex(index=cell_ids_arr)
assert list(meta.index) == list(cell_ids_arr)
counts_sparse.shape, meta.shape

((53513, 30314), (53513, 20))

In [6]:
adata = anndata.AnnData(
    X=counts_sparse,
    obs=meta,
    var=pd.DataFrame(index=gene_names_arr),
)
del counts_sparse
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 53513 × 30314
    obs: 'CellLine', 'Pool', 'Indication', 'Genes_expressed', 'Discrete_cluster_minpts5_eps1.8', 'Discrete_cluster_minpts5_eps1.5', 'Discrete_cluster_minpts5_eps1.2', 'CNA_subclone', 'SkinPig_score', 'EMTI_score', 'EMTII_score', 'EMTIII_score', 'IFNResp_score', 'p53Sen_score', 'EpiSen_score', 'StressResp_score', 'ProtMatu_score', 'ProtDegra_score', 'G1_S_score', 'G2_M_score'

In [7]:
numeric_score_cols = [c for c in adata.obs.columns if c.endswith('_score')]
for col in numeric_score_cols:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

for col in ['CellLine', 'Pool', 'Indication']:
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype(str)

if 'Genes_expressed' in adata.obs.columns:
    adata.obs['Genes_expressed'] = pd.to_numeric(adata.obs['Genes_expressed'], errors='coerce')

adata.obs.dtypes

CellLine                            object
Pool                                object
Indication                          object
Genes_expressed                      int64
Discrete_cluster_minpts5_eps1.8     object
Discrete_cluster_minpts5_eps1.5     object
Discrete_cluster_minpts5_eps1.2     object
CNA_subclone                        object
SkinPig_score                      float64
EMTI_score                         float64
EMTII_score                        float64
EMTIII_score                       float64
IFNResp_score                      float64
p53Sen_score                       float64
EpiSen_score                       float64
StressResp_score                   float64
ProtMatu_score                     float64
ProtDegra_score                    float64
G1_S_score                         float64
G2_M_score                         float64
dtype: object

Basic gene/cell filters, matching the thresholds used in the original F1L Kinker starter notebook (genes detected in fewer than 10 cells, and cells with fewer than 200 genes detected, are dropped as uninformative/low-quality).

In [8]:
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.filter_cells(adata, min_genes=200)
print(f"After filtering: {adata.n_obs} cells, {adata.n_vars} genes")
adata

Before filtering: 53513 cells, 30314 genes


After filtering: 53513 cells, 23081 genes


AnnData object with n_obs × n_vars = 53513 × 23081
    obs: 'CellLine', 'Pool', 'Indication', 'Genes_expressed', 'Discrete_cluster_minpts5_eps1.8', 'Discrete_cluster_minpts5_eps1.5', 'Discrete_cluster_minpts5_eps1.2', 'CNA_subclone', 'SkinPig_score', 'EMTI_score', 'EMTII_score', 'EMTIII_score', 'IFNResp_score', 'p53Sen_score', 'EpiSen_score', 'StressResp_score', 'ProtMatu_score', 'ProtDegra_score', 'G1_S_score', 'G2_M_score', 'n_genes'
    var: 'n_cells'

In [9]:
adata.write(os.path.join(OUTS_DIR, '01_kinker_raw.h5ad'))